# GRPO Training for Hierarchical Chain-of-Thought

Fine-tune the SFT-trained HCoT model using Group Relative Policy Optimization (GRPO).

**Reward functions** (defined in `trainer/rewards.py`):
1. **Correctness** — is the `\boxed{}` answer correct? (weight 2.0)
2. **Syntax** — are `[THOUGHT]`/`[SOLUTION]`/`[RETURN]` blocks well-formed? (weight 1.0)
3. **Leaf length** — penalise overly long leaf thoughts (weight 0.5)
4. **Tree depth** — penalise excessive nesting (weight 0.5)
5. **Compression ratio** — reward good solution-to-thought ratio (weight 0.5)
6. **Format** — check `<think>`, `</think>`, `\boxed{}`, and block presence (weight 0.5)

## Setup

In [3]:
import sys, os

# When running in Colab, clone the repo and add lib/ to the path
if "google.colab" in sys.modules:
    if not os.path.exists("cognitive-compression"):
        !git clone https://github.com/anujjamwal/cognitive-compression.git

    !cd cognitive-compression && git branch --set-upstream-to=origin/prune-aware-training prune-aware-training && git pull
    sys.path.insert(0, "cognitive-compression/lib")
    !pip install trl peft math-verify wandb
else:
    # Local: notebook lives inside lib/ already
    sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
    !pip install trl peft math-verify wandb

  Using cached wandb-0.25.0-py3-none-manylinux_2_28_x86_64.whl.metadata (12 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.2-py3-none-any.whl.metadata (4.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 12.1 MB/s  0:00:00
Using cached wandb-0.25.0-py3-none-manylinux_2_28_x86_64.whl (25.3 MB)
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
Using cached pydantic_core-2.41.5-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached gitpython-3.1.46-py3-none-a

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig

from trainer import prepare_base_model
from trainer.rewards import (
    correctness_reward,
    syntax_reward,
    leaf_length_reward,
    depth_reward,
    compression_reward,
    format_reward,
)
from custom_generate import generate

# W&B for experiment tracking
import wandb

In [5]:
from huggingface_hub import login as notebook_login
notebook_login()

wandb.login()
wandb.init(
    project="hcot-grpo",
    config={
        "model": "OpenMath-Nemotron-1.5B-hcot",
        "method": "GRPO",
        "num_generations": 4,
        "reward_weights": [2.0, 1.0, 0.5, 0.5, 0.5, 0.5],
    },
)

ImportError: The `notebook_login` function can only be used in a notebook (Jupyter or Colab) and you need the `ipywidgets` module: `pip install ipywidgets`.

## Model & Dataset

In [6]:
MODEL_NAME = "anujjamwal/OpenMath-Nemotron-1.5B-PruneAware"
GRPO_REPO_ID = "anujjamwal/OpenMath-Nemotron-1.5B-PruneAware-grpo"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 509.78it/s]


In [7]:
DATASET_NAME = "anujjamwal/OpenMathReasoning-Sampled-Hierarchical-Cot"
SYSTEM_PROMPT = "Solve the following math problem. Make sure to put the answer (and only answer) inside \\boxed{}."

raw_dataset = load_dataset(DATASET_NAME, split="train").filter(
    lambda ex: len(ex["hierarchical_cot"]) > 50
)
print(f"Dataset size: {len(raw_dataset)}")
print(f"Columns: {raw_dataset.column_names}")

Filter: 100%|██████████| 1502/1502 [00:00<00:00, 28207.67 examples/s]

Dataset size: 1499
Columns: ['id', 'question', 'expected_answer', 'problem_source', 'generated_solution', 'pass_rate_72b_tir', 'used_in_kaggle', 'hcot_model', 'hierarchical_cot', 'hierarchical_cot_raw']


In [13]:
def to_grpo_format(example):
    """Convert dataset example to GRPO conversational format.

    Returns a `prompt` (list of message dicts for system + user) and
    keeps `expected_answer` so it is passed as a kwarg to reward functions.
    """
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["question"]},
        ],
        "expected_answer": example["expected_answer"],
    }


dataset = raw_dataset.map(
    to_grpo_format,
    remove_columns=[c for c in raw_dataset.column_names if c != "expected_answer"],
)
print(f"GRPO dataset size: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"Sample prompt: {dataset[0]['prompt']}")

Map: 100%|██████████| 1499/1499 [00:00<00:00, 11480.14 examples/s]

GRPO dataset size: 1499
Columns: ['expected_answer', 'prompt']
Sample prompt: [{'content': 'Solve the following math problem. Make sure to put the answer (and only answer) inside \\boxed{}.', 'role': 'system'}, {'content': 'Find the domain of the function \\( g(x) = \\int_{10}^{x} \\log_{10}(\\log_{10}(t^2 - 1000t + 10^{1000})) \\, dt \\).', 'role': 'user'}]


## Reward Functions

All six reward functions are defined in `trainer/rewards.py`. Quick sanity check below.

In [ ]:
# --- Sanity check reward functions on synthetic examples ---

good_completion = [
    {"role": "assistant", "content": (
        "<think>\n"
        "[THOUGHT] We need to solve 2+2.\n"
        "[THOUGHT] 2+2 = 4. [SOLUTION] The sum is 4. [RETURN]\n"
        "[SOLUTION] The answer is 4. [RETURN]\n"
        "</think>\n"
        "\\boxed{4}"
    )}
]

bad_syntax = [
    {"role": "assistant", "content": (
        "<think>\n"
        "[THOUGHT] unclosed block\n"
        "</think>\n"
        "\\boxed{4}"
    )}
]

no_format = [
    {"role": "assistant", "content": "The answer is 4."}
]

test_completions = [good_completion, bad_syntax, no_format]
test_answers = ["4", "4", "4"]

print("Correctness :", correctness_reward(test_completions, expected_answer=test_answers))
print("Syntax      :", syntax_reward(test_completions))
print("Leaf length :", leaf_length_reward(test_completions))
print("Depth       :", depth_reward(test_completions))
print("Compression :", compression_reward(test_completions))
print("Format      :", format_reward(test_completions))

## PEFT / LoRA (optional)

Set `USE_LORA = True` to train with LoRA instead of full fine-tuning. This reduces trainable parameters from ~1.5B to ~10M, dramatically cutting optimizer memory and enabling cheaper GPUs.

In [ ]:
USE_LORA = False

peft_config = None
if USE_LORA:
    from peft import LoraConfig
    peft_config = LoraConfig(
        r=32,
        lora_alpha=64,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05,
        task_type="CAUSAL_LM",
    )
    print(f"LoRA enabled: ~{sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M trainable params")
else:
    print("Full fine-tuning enabled")

## Training

In [ ]:
# TensorBoard (optional — W&B is also enabled for remote monitoring)
%load_ext tensorboard
%tensorboard --logdir ./hcot-nemotron-1.5b-grpo

In [ ]:
training_args = GRPOConfig(
    output_dir="./hcot-nemotron-1.5b-grpo",
    hub_model_id=GRPO_REPO_ID,

    # Generation
    num_generations=4,
    max_completion_length=4096,
    generation_kwargs={
        "processing_class": tokenizer,
        "return_unpruned_output": True,
    },

    # Optimisation
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-7,
    beta=0.1,
    bf16=True,
    gradient_checkpointing=True,
    mask_truncated_completions=True,
    torch_compile=True,

    # Reward weighting: correctness >> syntax >> structural rewards
    reward_weights=[2.0, 1.0, 0.5, 0.5, 0.5, 0.5],

    # Logging & checkpointing
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    report_to=["tensorboard", "wandb"],
    push_to_hub=False,
)

In [10]:
from transformers import TrainerCallback


class WandbSampleCallback(TrainerCallback):
    """Log sample completions and per-reward breakdowns to W&B every N steps."""

    def __init__(self, tokenizer, reward_funcs, dataset, every_n_steps=50, num_samples=3):
        self.tokenizer = tokenizer
        self.reward_funcs = reward_funcs
        self.reward_names = [
            "correctness", "syntax", "leaf_length",
            "depth", "compression", "format",
        ]
        self.dataset = dataset
        self.every_n_steps = every_n_steps
        self.num_samples = num_samples

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % self.every_n_steps != 0 or model is None:
            return

        model.eval()
        samples = self.dataset.select(range(min(self.num_samples, len(self.dataset))))
        table = wandb.Table(columns=[
            "step", "prompt", "completion", "expected_answer",
            *self.reward_names, "total_reward",
        ])

        for i, sample in enumerate(samples):
            inputs = self.tokenizer.apply_chat_template(
                sample["prompt"],
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            ).to(model.device)

            with torch.no_grad():
                gen = model.generate(**inputs, max_new_tokens=2048, do_sample=True, temperature=0.7)

            completion_text = self.tokenizer.decode(
                gen[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=False
            )
            completion_msg = [{"role": "assistant", "content": completion_text}]

            # Score with each reward function
            scores = {}
            for name, fn in zip(self.reward_names, self.reward_funcs):
                if name == "correctness":
                    r = fn([completion_msg], expected_answer=[sample["expected_answer"]])
                else:
                    r = fn([completion_msg])
                scores[name] = r[0]

            weights = [2.0, 1.0, 0.5, 0.5, 0.5, 0.5]
            total = sum(w * scores[n] for w, n in zip(weights, self.reward_names))
            prompt_text = sample["prompt"][-1]["content"][:200]

            table.add_data(
                state.global_step, prompt_text, completion_text[:1000],
                sample["expected_answer"],
                *[scores[n] for n in self.reward_names], total,
            )

        wandb.log({"sample_completions": table}, step=state.global_step)
        model.train()

In [15]:
reward_funcs = [
    correctness_reward,
    syntax_reward,
    leaf_length_reward,
    depth_reward,
    compression_reward,
    format_reward,
]

sample_callback = WandbSampleCallback(
    tokenizer=tokenizer,
    reward_funcs=reward_funcs,
    dataset=dataset,
    every_n_steps=50,
    num_samples=3,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_funcs,
    args=training_args,
    train_dataset=dataset,
    callbacks=[sample_callback],
    peft_config=None,
)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 411.13it/s]


In [16]:
torch.cuda.empty_cache()
trainer.train()

TypeError: Object of type Qwen2Tokenizer is not JSON serializable

In [ ]:
trainer.push_to_hub()
tokenizer.push_to_hub(GRPO_REPO_ID)
wandb.finish()

## Test Generation

In [ ]:
from custom_generate import generate

model.eval()

sample_prompt = dataset[10]["prompt"]
inputs = tokenizer.apply_chat_template(
    sample_prompt,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

gen_out = model.generate(
    **inputs,
    max_new_tokens=4096,
    prune_aware=True,
    processing_class=tokenizer,
    custom_generate=generate._sample,
)

output = tokenizer.decode(gen_out[0], skip_special_tokens=False)
print(output.replace("\\n", "\n"))
print(f"\nOutput tokens: {gen_out.shape[-1]}")
print(f"Expected answer: {dataset[10]['expected_answer']}")